In [ ]:
# @title 1) Setup Environment
import sys, os, shutil
from google.colab import drive

#%pip install -q --upgrade --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!apt -y install -qq aria2

# Clone the OFFICIAL ComfyUI repository
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt

# GGUF custom nodes from city96
!git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI_GGUF/requirements.txt

comfy_path = "/content/ComfyUI"
if not comfy_path in sys.path:
  sys.path.insert(0, comfy_path)

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

# Copy framework from google drive
gdrive_code_path = "/content/drive/MyDrive/AI/Colab/gen_ai/faga"
local_code_path = "/content/gen_ai"
if os.path.exists(local_code_path):
  shutil.rmtree(local_code_path)
shutil.copytree(
  gdrive_code_path,
  os.path.join(local_code_path, "faga"),
  ignore=lambda path, names:
  ([n for n in names if not os.path.isdir(os.path.join(path, n)) and not n.lower().endswith(".py")])
)

if not local_code_path in sys.path:
  sys.path.insert(0, local_code_path)

print("✅ Environment Setup Complete!")

In [ ]:
# @title 2) Display Form
import platform
from faga.ui.prompt_form import PromptForm
from faga.ui.ui_helper import UiHelper
from faga.ai.file_manager import FileManager
from faga.ai.model_params import ModelParams
from faga.ai.workflow import Workflow
import ipywidgets as widgets

fm = FileManager("." if platform.system() == "Windows" else "/content/drive/MyDrive/AI/Colab/gen_ai")

msg_out: widgets.Output = widgets.Output(layout={'overload': 'auto'})
msg_out.add_class("box_container")


def on_generate(model_id: str, prompt: ModelParams):
  if (images := Workflow.generate(fm, model_id, prompt, "/content/ComfyUI", msg_out)) is not None:
    helper = UiHelper(images, prompt, msg_out)
    for image in helper.get_images():
      with msg_out:
        display(image)


form: PromptForm = PromptForm(fm, on_generate, msg_out)
display(form.get_style())
display(form.get_header_control())
display(form.get_body_control())
display(msg_out)

In [ ]:
# @title 3) Clean memory and re-import

UiHelper.clean_memory()
UiHelper.force_reimports()